# Signal quality report

Load a recording by path, compute quality metrics over a shared grid of intervals,
apply filters to the joined table, then visualize whatever got flagged.

The pipeline is four independent stages, which is the point of the library:

| stage | what it does | what it does *not* do |
|---|---|---|
| `load` | path → `Recording` (xarray) | interpret anything |
| `compute` | metrics → one joined table | decide what is bad |
| `apply_filters` | table → flags | recompute anything |
| `viz` | flags → evidence | hide what it excluded |

Because filtering is separate from measurement, you can re-judge the same table
under a different policy without recomputing a thing.

> **Data stays outside this repo.** Point `STUDY` at a recording on disk. Before
> sharing this notebook, clear outputs or de-identify them — channel names and
> annotations can carry identifying information.
## The example in this notebook

By default this runs on a **synthetic recording** built by
`sq.make_demo_recording()`, which injects a known fault of every kind the
library checks for — mains pickup, an attenuated contact, a dead electrode, an
isolated one, converter clipping, muscle contamination, a recording gap, and a
corrupted acquisition clock. Because the generator returns its own ground
truth, §4.1 *scores* the detections instead of asking you to eyeball them.

Point `STUDY` at a real recording and set `USE_SYNTHETIC = False` to run the
identical pipeline on real data.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import signal_quality as sq
from signal_quality import metrics as M

%matplotlib inline
plt.rcParams["figure.figsize"] = (12, 5)
pd.set_option("display.width", 200)

# The synthetic example runs anywhere and carries known faults, so the whole
# report can be demonstrated (and scored) without touching patient data.
# Set USE_SYNTHETIC = False and point STUDY at a real recording to use one.
USE_SYNTHETIC = True
STUDY = Path("~/Documents/remy_eeg_northwell_2026-06-23/eeg_open_source/remy_eeg_lossless.h5").expanduser()
LINE_FREQ = 60.0  # 60 in the Americas, 50 across most of the rest of the world

## 1. Load

`load()` sniffs the path. XLTEK studies and lossless HDF5 archives keep their raw
ADC counts and acquisition stamp tables, which is what makes clipping and
timestamp checks possible; other formats load fine but those checks report
themselves unavailable rather than guessing.

In [ ]:
if USE_SYNTHETIC:
    rec, truth = sq.make_demo_recording(line_freq=LINE_FREQ)
    print("synthetic demo recording with known injected faults")
else:
    rec, truth = sq.load(STUDY, line_freq=LINE_FREQ), None

print(rec)
print("raw counts available:", rec.has_counts)
print("stamp tables available:", "stamps" in rec.provenance)
rec.ds

## 1.1 Look at the raw signals first

Before any metric runs. Every number later in this notebook is a summary, and
summaries only answer questions you already thought to ask — a look at the
actual signal is what catches the thing no threshold was written for.

Two details make this readable on a long recording:

* Each pixel column shows the **min–max envelope** of the samples it covers, not
  a decimated sample. Plain decimation would drop exactly the brief excursions
  that are usually the most diagnostic marks on the page.
* Traces are **clipped to their lane**, and any channel that overruns it is
  named underneath with its true peak. One channel swinging to the converter
  rail is thousands of times larger than EEG and would otherwise squash
  everything else into flat lines.

Even at this glance you should be able to pick out the dead channels, the
attenuated one, the saturating one, and the recording gap.

* **Mixed channel types are normalised per channel.** EEG in microvolts and a
  DC or position channel in device units cannot share one absolute scale — the
  DC channels would set the spacing and every EEG trace would collapse to a
  hairline. Lanes are then in each channel's own SD, and the scale bar says so.
  Pass `normalize=False` to force absolute µV when all channels are comparable.


In [ ]:
# All channels, whole recording. Per-channel SD scaling when types are mixed.
sq.viz.plot_overview(rec)
plt.tight_layout()
plt.show()

# EEG only, on a shared absolute µV scale — amplitudes are comparable here,
# so an attenuated or oversized electrode stands out directly.
sq.viz.plot_overview(rec.pick(ch_type="eeg"), normalize=False)
plt.tight_layout()
plt.show()

A closer look at a few seconds, at a stated µV scale — the scale bar and the
lane spacing in the axis label are what let you judge amplitude, since the y
ticks carry channel names rather than numbers.

In [ ]:
eeg_names = rec.pick(ch_type="eeg").ch_names
sq.viz.plot_channel_snippet(rec, eeg_names[:10], t_start=20.0, duration=8.0)
plt.tight_layout()
plt.show()

## 2. Generic integrity checks

These are recording-scope and signal-agnostic — they ask whether the data exists,
whether its clock is coherent, and whether the channels can honestly share one
time axis. They are kept in their own table rather than broadcast across every
channel row, because a gap is a property of the recording, not of any channel.

In [ ]:
issues = sq.check_integrity(rec)
issues

In [ ]:
for issue in issues.detail.unique():
    print(issue)
    print()

In [ ]:
sq.viz.plot_availability(rec, issues)
plt.show()

## 3. Compute metrics over a shared interval grid

Every metric is computed over the *same* `IntervalGrid`, so the results share a
row index exactly and the join is a plain concat — no alignment guesswork, no
NaN padding.

`IntervalGrid.whole` gives one row per channel (a whole-recording summary);
`IntervalGrid.fixed(rec, 30.0)` gives a per-window trend. Section 6 uses the
windowed form for the same metrics.

In [ ]:
grid = sq.IntervalGrid.whole(rec)

mf = sq.compute(
    [
        M.RMS(),              # broadband amplitude
        M.LineRatio(),        # mains pickup -> contact quality
        M.EMGFraction(),      # muscle contamination
        M.MaxCorrelation(),   # isolation
        M.FlatFraction(),     # dead / flat stretches
        M.ClipFraction(),     # converter saturation (needs raw counts)
    ],
    rec,
    grid,
    ch_type="eeg",
)

for note in mf.notes:  # metrics that could not be computed, and why
    print("note:", note)

mf.table.droplevel("interval").drop(columns=["t_start", "t_end", "coverage"]).sort_values(
    "line_ratio", ascending=False
).round(2)

## 4. Apply filters

Thresholds live here, separate from the measurements, so re-judging costs
nothing. `DEFAULT_FILTERS` carries the defaults tuned for clinical scalp EEG —
treat them as a starting point for any other modality.

Each flag records the value and threshold that produced it, so no verdict is
unfalsifiable.

In [ ]:
flags = sq.apply_filters(mf, sq.DEFAULT_FILTERS)
verdicts = sq.verdict(flags, channels=mf.table.index.get_level_values("channel").unique())

print(verdicts["verdict"].value_counts().to_dict())
verdicts[verdicts["verdict"] != "good"]

In [ ]:
# Which issues are present, and how widespread each one is.
flags.groupby(["flag", "severity"])["channel"].agg(["nunique", lambda s: ", ".join(sorted(set(s)))]).rename(
    columns={"nunique": "n_channels", "<lambda_0>": "channels"}
)

### 4.1 Scoring against ground truth

Only possible for the synthetic example, and the reason it exists: the generator
records which fault it placed on which channel, so detection can be *measured*
rather than admired.

Two things are worth reading carefully. **Extra flags on an already-broken
channel are not false positives** — a dead electrode genuinely correlates with
nothing, and a channel swinging to the converter rail genuinely is uncorrelated
with the montage, so both legitimately trip `ISOLATED` as well. What would be
damning is a flag on a channel with no injected fault at all. Second, `OSAT` is
an auxiliary channel and never enters the EEG metric table; it is caught by the
integrity pass in §2 as a `dead_channel`, which is why it does not appear here.

In [ ]:
if truth is None:
    print("real recording — no ground truth to score against")
else:
    eeg = set(mf.table.index.get_level_values("channel"))
    detected = flags.groupby("channel")["flag"].agg(lambda s: "+".join(sorted(set(s))))

    score = truth.loc[[c for c in truth.index if c in eeg]].copy()
    score["detected"] = detected.reindex(score.index).fillna("")

    def outcome(row):
        want = set(row["injected"].split("+")) - {""}
        got = set(row["detected"].split("+")) - {""}
        if not want:
            return "clean" if not got else "FALSE POSITIVE: " + "+".join(sorted(got))
        if want <= got:
            extra = got - want
            return "caught" + (f"  (also {'+'.join(sorted(extra))})" if extra else "")
        return "MISSED: " + "+".join(sorted(want - got))

    score["result"] = score.apply(outcome, axis=1)

    n_injected = int((score["injected"] != "").sum())
    n_caught = int(score["result"].str.startswith("caught").sum())
    n_false = int(score["result"].str.startswith("FALSE").sum())
    print(f"injected faults detected : {n_caught}/{n_injected}")
    print(f"false positives on clean channels: {n_false}")
    display(score[(score["injected"] != "") | (score["detected"] != "")])

## 5. Visualize what was flagged

### 5.1 Electrode positions on the head model

Left: the metric interpolated across the scalp — illustrative, since only the
electrode sites carry measurements. Right: the authoritative per-electrode
verdict, unsmoothed, labelled with the flags that fired.

In [ ]:
sq.viz.plot_contact_quality(mf, verdicts, metric="line_ratio", log=True)
plt.show()

### 5.2 Spectra: flagged vs unflagged channels

A table saying a channel has a mains ratio of 900 is an assertion; a spectrum
showing its line-frequency spike towering over every clean channel is the
evidence. Check every flag class this way before acting on it.

In [ ]:
if "LINE_NOISE" in set(flags["flag"]):
    sq.viz.plot_good_bad_psd(rec, flags, flag="LINE_NOISE")
    plt.show()
else:
    print("no line-noise flags; showing the spread instead")
    sq.viz.plot_psd_examples(rec, mf, metric="line_ratio")
    plt.show()

### 5.3 The raw signal behind the flags

Worst flagged channels above, best unflagged below, same window and same gain.
This is where a "flat" channel that actually has a visible trace, or a clipping
flag driven by three stray samples, gives itself away.

In [ ]:
present = [f for f in ["LINE_NOISE", "AMP_OUTLIER", "FLAT", "CLIPPING"] if f in set(flags["flag"])]
metric_for = {
    "LINE_NOISE": "line_ratio",
    "AMP_OUTLIER": "rms",
    "FLAT": "flat_frac",
    "CLIPPING": "clip_pct",
}
for flag in present[:2]:
    sq.viz.plot_flagged_examples(rec, mf, flags, flag=flag, metric=metric_for[flag], n=3)
    plt.show()

### 5.4 Highly correlated pairs

A shortlist, not a verdict. Near-unity correlation is consistent with a salt
bridge, but on a common-reference recording the shared reference and drift push
almost every pair toward 1 — which is exactly why this library ships no
automatic bridge filter. Confirm a candidate against a bipolar derivation: a
bridged pair collapses to a much smaller amplitude than a healthy one.

In [ ]:
M.correlation_pairs(rec, threshold=0.95, top=10)

## 6. Quality over time

The same metrics, recomputed on a windowed grid. Nothing about the metrics or
filters changes — only the grid does. A recording can be perfectly continuous
and still be unusable for stretches, which is what the artifact trace shows.

The trend aggregates with `max` rather than `median`: a handful of bad electrodes
is exactly what this is meant to surface, and the median across a mostly-clean
montage would hide them. Intervals inside the recording gap are left blank — there
is no data there to be clean or dirty.


In [ ]:
wgrid = sq.IntervalGrid.fixed(rec, 30.0)
wmf = sq.compute([M.RMS(), M.LineRatio(), M.PeakToPeak()], rec, wgrid, ch_type="eeg")

fig, axes = plt.subplots(2, 1, figsize=(13, 7))
sq.viz.plot_metric_trend(wmf, "line_ratio", rec=rec, ax=axes[0],
                         agg="max", threshold=300)
sq.viz.plot_clean_fraction(wmf, rec=rec, metric="p2p", threshold=150.0, ax=axes[1])
fig.tight_layout()
plt.show()

## 7. Summary

Everything below is derived from the signal itself. Use it to decide which
channels to exclude from quantitative analysis — and note that excluding them
is a judgement call the library deliberately leaves to you.

In [ ]:
bad = verdicts[verdicts["verdict"] == "bad"].index.tolist()
marginal = verdicts[verdicts["verdict"] == "marginal"].index.tolist()

print(f"recording      : {rec}")
print(f"missing data   : {100 * (1 - rec.covered.mean()):.1f}% of the timeline")
print(f"integrity      : {len(issues)} finding(s) — {sorted(set(issues['check'])) if len(issues) else 'none'}")
print(f"bad channels   : {len(bad)} — {', '.join(bad) if bad else 'none'}")
print(f"marginal       : {len(marginal)} — {', '.join(marginal) if marginal else 'none'}")
print()
print("Suggested exclusion list for quantitative analysis:")
print("  BAD =", bad)